In [ ]:
Here is the English version of the program, including the class definitions, the Pandas-based database manager, and the user interface functions.

🚀 Python Solution (with Pandas) - English Version

This solution uses a class EM_Record to represent a single session and a class EM_Database to manage the data using a Pandas DataFrame.

1. Data Structures and EM_Database Class

Python

import pandas as pd
from datetime import datetime

# Definition of possible options for data consistency
POSSIBLE_MICROSCOPES = ["Microscope A", "Microscope B", "Microscope C"]
POSSIBLE_ANALYSES = ["EDX", "EBSD", "WDS", "SE/BSE Imaging"]

class EM_Record:
    """Represents a single work record on the electron microscope."""
    def __init__(self, name, date, microscope, sample, analyses, completed=False):
        self.name = name
        self.date = date
        self.microscope = microscope
        self.sample = sample
        self.analyses = analyses
        self.completed = completed

    def __str__(self):
        status = "Yes" if self.completed else "No"
        return f"[{self.date}] {self.name} on {self.microscope} (Sample: {self.sample}, Analyses: {', '.join(self.analyses)}, Completed: {status})"

class EM_Database:
    """Manages the database of microscope work records using a Pandas DataFrame."""
    def __init__(self):
        # Initialize an empty DataFrame
        self.columns = ["Name", "Date", "Microscope", "Sample", "Analyses", "Completed"]
        self.df = pd.DataFrame(columns=self.columns)

    def add_record(self, record: EM_Record):
        """Adds a new record to the DataFrame."""
        # Create a new row as a Series
        new_row = pd.Series({
            "Name": record.name,
            "Date": record.date,
            "Microscope": record.microscope,
            "Sample": record.sample,
            # Store analyses as a comma-separated string for simpler display
            "Analyses": ", ".join(record.analyses),
            "Completed": "Yes" if record.completed else "No"
        })
        
        # Add the new row to the DataFrame
        # We use pd.concat to add the row, then reset the index
        self.df = pd.concat([self.df, new_row.to_frame().T], ignore_index=True)
        print("✅ Record successfully added.")

    def display_table(self):
        """Displays the entire DataFrame."""
        if self.df.empty:
            print("Database is empty.")
            return
        # Set options to display all columns/rows
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        print("\n--- 📋 OVERVIEW OF ALL RECORDS ---")
        print(self.df)
        print("------------------------------------\n")

    def filter_records(self, **kwargs):
        """
        Filters the DataFrame based on the specified criteria (e.g., name='Peter K.', completed='No').
        """
        if self.df.empty:
            print("Database is empty, nothing to filter.")
            return

        filtered_df = self.df.copy()

        # Applying filters
        for column, value in kwargs.items():
            # Ensure the column name exists in the DataFrame
            if column in self.df.columns:
                # Use .str.contains() for filtering analyses, which can have multiple values
                if column == "Analyses":
                    filtered_df = filtered_df[filtered_df[column].str.contains(value, case=False, na=False)]
                else:
                    # Standard filtering for other columns
                    filtered_df = filtered_df[filtered_df[column] == value]
            else:
                print(f"Warning: Column '{column}' does not exist.")

        if filtered_df.empty:
            print(f"\n❌ No records found for the specified criteria ({kwargs}).")
        else:
            print(f"\n--- 🔎 FILTERING RESULT ({kwargs}) ---")
            print(filtered_df)
            print("-------------------------------------------\n")
            
        return filtered_df

2. User Interface and Testing Implementation

Python

def input_record():
    """Gets inputs from the user and creates an EM_Record object."""
    print("\n--- 📝 ENTERING NEW RECORD ---")
    
    # 1. Name and Date
    name = input("Worker's Name: ")
    date = datetime.now().strftime("%Y-%m-%d") # Automatic setting of today's date
    print(f"Work Date: {date}")
    
    # 2. Microscope Selection
    print("\n--- Available Microscopes ---")
    for i, m in enumerate(POSSIBLE_MICROSCOPES):
        print(f"{i+1}. {m}")
    while True:
        try:
            selection = int(input(f"Select microscope number (1-{len(POSSIBLE_MICROSCOPES)}): "))
            if 1 <= selection <= len(POSSIBLE_MICROSCOPES):
                microscope = POSSIBLE_MICROSCOPES[selection - 1]
                break
            else:
                print("Invalid selection.")
        except ValueError:
            print("Please enter a number.")
            
    # 3. Sample
    sample = input("Sample Name/ID: ")
    
    # 4. Analyses Selection
    selected_analyses = []
    print("\n--- Available Analyses (can select multiple) ---")
    for i, a in enumerate(POSSIBLE_ANALYSES):
        print(f"{i+1}. {a}")
    
    while True:
        inputs = input(f"Enter the numbers of the analyses separated by commas (e.g., 1,3,4) or press Enter to finish: ")
        if not inputs:
            if not selected_analyses:
                 print("⚠️ You must select at least one analysis!")
                 continue
            break
            
        try:
            numbers = [int(c.strip()) for c in inputs.split(',')]
            valid_numbers = True
            for number in numbers:
                if 1 <= number <= len(POSSIBLE_ANALYSES):
                    analysis = POSSIBLE_ANALYSES[number - 1]
                    if analysis not in selected_analyses:
                        selected_analyses.append(analysis)
                else:
                    print(f"Invalid analysis number: {number}")
                    valid_numbers = False
            if valid_numbers:
                break
        except ValueError:
            print("Please enter numbers separated by commas.")
            
    # 5. Completion Status
    while True:
        completed_input = input("Was the work completed? (Y/N): ").upper()
        if completed_input in ['Y', 'YES']:
            completed = True
            break
        elif completed_input in ['N', 'NO']:
            completed = False
            break
        else:
            print("Invalid input. Enter 'Y' for Yes or 'N' for No.")

    return EM_Record(name, date, microscope, sample, selected_analyses, completed)

# --- Main Program Part ---
def main():
    database = EM_Database()
    
    # 1. Add initial test data
    database.add_record(EM_Record("Peter K.", "2025-11-20", "Microscope A", "Smpl. 101", ["EDX", "EBSD"], True))
    database.add_record(EM_Record("Mária V.", "2025-11-21", "Microscope B", "Smpl. 204", ["SE/BSE Imaging"], False))
    database.add_record(EM_Record("Ján S.", "2025-11-21", "Microscope A", "Smpl. 102", ["EDX", "WDS"], False))
    database.add_record(EM_Record("Peter K.", "2025-11-22", "Microscope C", "Smpl. 301", ["EBSD"], True))
    
    while True:
        print("\n==================================")
        print("🔬 EM Workplace Database - MENU")
        print("==================================")
        print("1. Add New Record")
        print("2. Display All Records")
        print("3. Filter Records")
        print("4. Exit Program")
        
        choice = input("Enter option number: ")
        
        if choice == '1':
            try:
                new_record = input_record()
                database.add_record(new_record)
            except Exception as e:
                print(f"An error occurred while entering the record: {e}")
                
        elif choice == '2':
            database.display_table()
            
        elif choice == '3':
            print("\n--- ⚙️ FILTERING OPTIONS (Enter value or leave blank) ---")
            
            filter_args = {}
            
            # Filter by Name
            name = input("Filter by Name (e.g., Peter K.): ")
            if name:
                filter_args['Name'] = name
            
            # Filter by Microscope
            microscope_input = input(f"Filter by Microscope ({'/'.join(POSSIBLE_MICROSCOPES)}): ")
            if microscope_input and microscope_input in POSSIBLE_MICROSCOPES:
                filter_args['Microscope'] = microscope_input
            
            # Filter by Completed Status
            completion = input("Filter by Completed Status (Yes/No): ")
            if completion in ['Yes', 'No']:
                filter_args['Completed'] = completion
                
            # Filter by Analyses (if record contains the analysis)
            analysis_filter = input(f"Filter by Analysis (e.g., EDX): ")
            if analysis_filter:
                filter_args['Analyses'] = analysis_filter

            if filter_args:
                database.filter_records(**filter_args)
            else:
                print("No criteria specified for filtering.")

        elif choice == '4':
            print("Program terminated. Goodbye!")
            break
            
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main()